# Targeted estimation (TMLE) of smooth features of the density

A HAL density fit gives plug-in estimates of features such as the survival function, the mean and the median.
Targeting updates the fit along a few extra directions so that the estimate of the chosen feature solves its
efficient score equation; the targeted estimate (the TMLE) is then asymptotically efficient and comes with an
influence-curve standard error.

The targeting step integrates exactly, so when its coefficients stay inside their L1 bound the score equations
hold at the targeting points themselves: the TMLE of S(t) equals the empirical survival at t, the TMLE of the
mean equals the sample mean.

Targeting is cross-fitted by default. The tuner below also fits its chosen parameters on each of its training
folds, and `targeting_inputs` passes those fold fits to the learners. Each fold's fit is then targeted on its
own validation fold, with one targeting coefficient for all folds, and the SE comes from each observation's
influence curve under its own fold's targeted fit. This is *cross-fitted with tuning on all folds*: the bound M
was chosen with every fold, including the one a fit is later scored on. `cross_fit="none"` gives the
full-sample step; `"refit"` and `"nested"` are explained in the README.

In [1]:
import numpy as np
import pandas as pd
from scipy.integrate import trapezoid

from haldensity.datasets import TruncatedGMM
from haldensity.full_uni import OptunaHyperparameterTuner
from haldensity.full_uni.targeting import (
    DensityL2TargetLearner, EntropyTargetLearner, MedianTargetLearner, MomentsTargetLearner,
    SurvivalTargetLearner, density_l2_estimand_variance, entropy_estimand_variance, median_estimand_variance,
    moments_estimand_variance, survival_estimand_variance, targeting_inputs,
)

truth = TruncatedGMM(
    components=[{"mean": 0.35, "std": 0.1, "lower": 0.0, "upper": 1.0},
                {"mean": 0.65, "std": 0.05, "lower": 0.0, "upper": 1.0},
                {"mean": 0.9, "std": 0.2, "lower": 0.0, "upper": 1.0}],
    weights=[0.4, 0.4, 0.2],
)
np.random.seed(1)
data = pd.DataFrame({"W1": truth.generate_samples(400)})

tuner = OptunaHyperparameterTuner("CVXPYEstimator", data, random_state=0, silent=True, show_progress=False)
tuner.optimize(n_trials=30, show_progress=False)
est = tuner.fit_best_model()
inputs = targeting_inputs(est)        # intercept, powers and nonzero knot coefficients, with their knots
print(f"order k = {est.basis_order}, M = {est.norm_constraint:.4g}, {inputs['grid_points_hal_selected'].size} knots")
print(f"cross-fitting: the tuner's {inputs['folds'].n_folds} fold fits")

order k = 2, M = 749.3, 7 knots
cross-fitting: the tuner's 5 fold fits


## Survival at chosen points

Choose targeting points inside the data. Each learner is built with the fit's order; `norm_constraint` bounds
the L1 norm of the targeting coefficients.

In [2]:
x = data["W1"].to_numpy()
points = np.array([0.3, 0.5, 0.7])
learner = SurvivalTargetLearner(norm_constraint=100, basis_order=est.basis_order)
fit = learner.run_m_step(uncensored_augmented=data, **inputs, targeting_points=points)
se = np.sqrt(survival_estimand_variance(targeted_fit=fit, uncensored_data=data, targeting_points=points))

grid = np.linspace(0, 1, 200001)
true_survival = [trapezoid(truth.compute_density(grid) * (grid > t), grid) for t in points]
pd.DataFrame({"t": points, "truth": true_survival, "plug-in": fit["plug_in"], "TMLE": fit["estimate"],
              "empirical": [np.mean(x > t) for t in points], "SE": se,
              "95% lower": fit["estimate"] - 1.96 * se, "95% upper": fit["estimate"] + 1.96 * se}).round(4)

,t,truth,plug-in,TMLE,empirical,SE,95% lower,95% upper
0,0.3,0.8763,0.8642,0.8600,0.8600,0.0174,0.8259,0.8941
1,0.5,0.6196,0.6280,0.6125,0.6125,0.0245,0.5645,0.6605
2,0.7,0.2177,0.2336,0.2250,0.2250,0.0209,0.1840,0.2660


`score_residual` is the largest error in the score equations; it is near 1e-12 unless the targeting
coefficients reach their L1 bound.

In [3]:
print(f"largest score-equation error: {fit['score_residual']:.1e}; cross_fit = {fit['cross_fit']!r}, "
      f"{fit['n_folds']} folds")

largest score-equation error: 3.8e-14; cross_fit = 'tuner', 5 folds


## The mean and the median

In [4]:
mean_fit = MomentsTargetLearner(norm_constraint=100, basis_order=est.basis_order).run_m_step(
    uncensored_augmented=data, **inputs, x_moment=1)
median_fit = MedianTargetLearner(norm_constraint=100, basis_order=est.basis_order).run_m_step(
    uncensored_augmented=data, **inputs)
true_mean = trapezoid(grid * truth.compute_density(grid), grid)
cdf = np.cumsum(truth.compute_density(grid)) * (grid[1] - grid[0])
true_median = grid[np.searchsorted(cdf / cdf[-1], 0.5)]
pd.DataFrame({
    "truth": [true_mean, true_median],
    "plug-in": [mean_fit["plug_in"][0], median_fit["plug_in"][0]],
    "TMLE": [mean_fit["estimate"][0], median_fit["estimate"][0]],
    "empirical": [x.mean(), np.median(x)],
    "SE": [np.sqrt(moments_estimand_variance(targeted_fit=mean_fit, uncensored_data=data, x_moment=1))[0],
           np.sqrt(median_estimand_variance(targeted_fit=median_fit, uncensored_data=data))[0]],
}, index=["mean", "median"]).round(5)

,truth,plug-in,TMLE,empirical,SE
mean,0.55967,0.56038,0.56042,0.56042,0.01045
median,0.60837,0.60816,0.61375,0.61631,0.01062


The median learner takes one targeting step at the initial fit's median m, as in the paper's median example:
the targeted density satisfies F(m) = P_n(X < m) at m, and the targeted median is close to, but need not
equal, an empirical median.

## Entropy and the integral of the squared density

The entropy, -integral p log p, has influence curve -log p(x) - psi; the integral of p^2 has 2 (p(x) - psi). Their
influence curves depend on the density itself, so a targeting step moves along the influence curve of the density
it starts from. For the entropy one step solves the score equation; for the integral of p^2 one step leaves a
second-order remainder in P_n D*, and `max_steps` repeats the step until it is below 1e-10.

Targeting removes the first-order term of the estimate's error, not the second-order remainder of the fit
(for the entropy, a Kullback-Leibler divergence; for p^2, minus a squared L2 distance). In our simulations that
remainder was still visible at n <= 1,600, so these two estimates carry a small bias of known sign: up for the
entropy, down for the integral of p^2.

In [5]:
entropy_fit = EntropyTargetLearner(norm_constraint=100, basis_order=est.basis_order).run_m_step(
    uncensored_augmented=data, **inputs)
l2_fit = DensityL2TargetLearner(norm_constraint=100, basis_order=est.basis_order, max_steps=25).run_m_step(
    uncensored_augmented=data, **inputs)
density = truth.compute_density(grid)
true_entropy = -trapezoid(np.where(density > 0, density * np.log(np.where(density > 0, density, 1.0)), 0.0), grid)
true_l2 = trapezoid(density ** 2, grid)
pd.DataFrame({
    "truth": [true_entropy, true_l2],
    "plug-in": [entropy_fit["plug_in"][0], l2_fit["plug_in"][0]],
    "cross-fitted plug-in": [entropy_fit["plug_in_cross_fitted"][0], l2_fit["plug_in_cross_fitted"][0]],
    "TMLE": [entropy_fit["estimate"][0], l2_fit["estimate"][0]],
    "SE": [np.sqrt(entropy_estimand_variance(targeted_fit=entropy_fit, uncensored_data=data))[0],
           np.sqrt(density_l2_estimand_variance(targeted_fit=l2_fit, uncensored_data=data))[0]],
    "steps": [len(entropy_fit["theta_targeting_path"]), len(l2_fit["theta_targeting_path"])],
}, index=["entropy", "integral of p^2"]).round(5)

,truth,plug-in,cross-fitted plug-in,TMLE,SE,steps
entropy,-0.33405,-0.27738,-0.27843,-0.28279,0.03314,1
integral of p^2,1.70989,1.58815,1.59035,1.62810,0.09900,7
